In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FixDatabase") \
    .config("spark.master", "spark://spark-master:7077") \
    .config("spark.sql.catalog.hive", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hive.catalog-impl", "org.apache.iceberg.hive.HiveCatalog") \
    .config("spark.sql.catalog.hive.uri", "thrift://hive-metastore:9083") \
    .config("spark.sql.catalog.hive.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.jars", ",".join([
        "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
        "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
        "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
    ])) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.defaultCatalog", "hive") \
    .getOrCreate()

25/09/20 08:37:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
from minio import Minio
client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)
bucket = 'warehouse'
if not client.bucket_exists(bucket):
    client.make_bucket(bucket)

In [15]:
# Create database with explicit location
spark.sql("CREATE DATABASE IF NOT EXISTS test_db LOCATION 's3a://warehouse/test_db'")

DataFrame[]

In [14]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
|  test_db|
+---------+



In [13]:
# Create a sample Iceberg table
spark.sql("""
CREATE TABLE IF NOT EXISTS test_db.test_table (
    id INT,
    name STRING,
    created_at TIMESTAMP
)
USING iceberg
LOCATION 's3a://warehouse/test_db/test_table'
""")

DataFrame[]

In [7]:
# Insert sample data
spark.sql("""
INSERT INTO test_db.test_table VALUES
    (1, 'Alice', CURRENT_TIMESTAMP),
    (2, 'Bob', CURRENT_TIMESTAMP),
    (3, 'Charlie', CURRENT_TIMESTAMP)
""")

DataFrame[]

In [8]:
# Query the table
result = spark.sql("SELECT * FROM test_db.test_table")
result.show()

+---+-------+--------------------+
| id|   name|          created_at|
+---+-------+--------------------+
|  1|  Alice|2025-09-20 08:22:...|
|  2|    Bob|2025-09-20 08:22:...|
|  3|Charlie|2025-09-20 08:22:...|
|  1|  Alice|2025-09-20 08:38:...|
|  2|    Bob|2025-09-20 08:38:...|
|  3|Charlie|2025-09-20 08:38:...|
|  1|  Alice|2025-09-20 08:27:...|
|  2|    Bob|2025-09-20 08:27:...|
|  3|Charlie|2025-09-20 08:27:...|
|  1|  Alice|2025-09-20 08:35:...|
|  2|    Bob|2025-09-20 08:35:...|
|  3|Charlie|2025-09-20 08:35:...|
+---+-------+--------------------+



In [9]:
# Verify table metadata in Hive Metastore
tables = spark.sql("SHOW TABLES IN test_db")
tables.show()

+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|  test_db|test_table|      false|
+---------+----------+-----------+



In [10]:
spark.sql("SELECT * FROM test_db.test_table.snapshots").show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2025-09-20 08:22:...|6260898649158253464|               NULL|   append|s3a://warehouse/t...|{spark.app.id -> ...|
|2025-09-20 08:27:...| 825877854053591782|6260898649158253464|   append|s3a://warehouse/t...|{spark.app.id -> ...|
|2025-09-20 08:35:...|2268611663476647098| 825877854053591782|   append|s3a://warehouse/t...|{spark.app.id -> ...|
|2025-09-20 08:38:...|4115384562414966526|2268611663476647098|   append|s3a://warehouse/t...|{spark.app.id -> ...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+



In [16]:
spark.sql("SHOW DATABASES IN hive").show()

+---------+
|namespace|
+---------+
|  default|
|  test_db|
+---------+

